# Lab 2 · Ingest Healthy 365 files into **Bronze**

Load the uploaded Healthy 365 app files from `Files/landing/` into Delta **bronze** tables.
Bronze = raw, as-received. We clean and conform in Lab 3.

> **Attach** the `lh_resident360` Lakehouse to this notebook first (Explorer → **Add lakehouse**).

In [ ]:
from pyspark.sql import functions as F
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
LANDING = "Files/landing"

## 1. Meal logs (diet) — CSV with a few deliberate dirty rows

In [ ]:
meals = spark.read.option("header", True).csv(f"{LANDING}/meal_logs.csv")
meals.write.mode("overwrite").saveAsTable("bronze.h365_meal_logs")
print("bronze.h365_meal_logs:", spark.table("bronze.h365_meal_logs").count(), "rows")

## 2. Event & class bookings

In [ ]:
ev = spark.read.option("header", True).csv(f"{LANDING}/events_bookings.csv")
ev.write.mode("overwrite").saveAsTable("bronze.h365_event_bookings")
print("bronze.h365_event_bookings:", spark.table("bronze.h365_event_bookings").count(), "rows")

## 3. Programme enrolments (Healthier SG, EDSH, I Quit, …)

In [ ]:
pr = spark.read.option("header", True).csv(f"{LANDING}/programme_enrolments.csv")
pr.write.mode("overwrite").saveAsTable("bronze.h365_programme_enrolments")
print("bronze.h365_programme_enrolments:", spark.table("bronze.h365_programme_enrolments").count(), "rows")

## 4. Healthpoints ledger — JSON (multiline array)

In [ ]:
hp = spark.read.option("multiline", True).json(f"{LANDING}/rewards_healthpoints.json")
hp.write.mode("overwrite").saveAsTable("bronze.h365_rewards")
print("bronze.h365_rewards:", spark.table("bronze.h365_rewards").count(), "rows")

## 5. eVoucher redemptions (Healthpoints → merchant vouchers)

In [ ]:
vr = spark.read.option("header", True).csv(f"{LANDING}/evoucher_redemptions.csv")
vr.write.mode("overwrite").saveAsTable("bronze.h365_evoucher_redemptions")
print("bronze.h365_evoucher_redemptions:", spark.table("bronze.h365_evoucher_redemptions").count(), "rows")

## 6. Challenge participation (National Steps Challenge, Eat Drink Shop Healthy, …)

In [ ]:
ch = spark.read.option("header", True).csv(f"{LANDING}/challenges.csv")
ch.write.mode("overwrite").saveAsTable("bronze.h365_challenges")
print("bronze.h365_challenges:", spark.table("bronze.h365_challenges").count(), "rows")

## 7. Verify — six bronze tables landed

> **Note:** the mirrored Databricks estate (`hpb_databricks_mirror.gold.*`) is read directly — no copy into bronze needed. The next notebook adds an **external API** source.

In [ ]:
for t in ["h365_meal_logs","h365_event_bookings","h365_programme_enrolments",
          "h365_rewards","h365_evoucher_redemptions","h365_challenges"]:
    print(f"bronze.{t:32s}", spark.table(f"bronze.{t}").count(), "rows")